# Projeto: World Happiness Report - Pipeline de Dados
**Objetivo:** Transformar a base bruta (Raw) em uma base refinada e pronta para consumo no Power BI (Gold) e responder as seguintes perguntas:
Qual foi o pais com menor indice "Life Ladder"? 
Qual é a média de "Generosity" dos TOP 10 países, por ano? 
Qual foi o pais que teve queda de "Genorosity" ao longo dos anos ? Classificar o TOP 5
Que dado interessante você poderia acrescentar ao estudo?.
**Etapa 1:** Importação de bibliotecas e carregamento dos dados brutos.



In [52]:
# !pip install pandas -q

# Importando a biblioteca pandas (nossa principal ferramenta de manipulação de dados)
import pandas as pd

# Carregando o arquivo bruto (Camada Raw)
# Utilizamos barras normais (/) para compatibilidade entre diferentes sistemas operacionais
caminho_arquivo = 'Data/world-happiness-report.csv'
df_raw = pd.read_csv(caminho_arquivo)

# Exibindo as 5 primeiras linhas para visualização da estrutura
print("--- Amostra da Base de Dados (Head) ---")
display(df_raw.head())

# Exibindo um resumo técnico: total de linhas, colunas, tipos de dados e valores nulos
print("\n--- Informações Técnicas da Tabela (Info) ---")
df_raw.info()

--- Amostra da Base de Dados (Head) ---


,Country name,year,Life Ladder,Log GDP per capita,Social support,Healthy life expectancy at birth,Freedom to make life choices,Generosity,Perceptions of corruption,Positive affect,Negative affect
0,Afghanistan,2008,3.724,7.370,0.451,50.80,0.718,0.168,0.882,0.518,0.258
1,Afghanistan,2009,4.402,7.540,0.552,51.20,0.679,0.190,0.850,0.584,0.237
2,Afghanistan,2010,4.758,7.647,0.539,51.60,0.600,0.121,0.707,0.618,0.275
3,Afghanistan,2011,3.832,7.620,0.521,51.92,0.496,0.162,0.731,0.611,0.267
4,Afghanistan,2012,3.783,7.705,0.521,52.24,0.531,0.236,0.776,0.710,0.268



--- Informações Técnicas da Tabela (Info) ---
<class 'pandas.DataFrame'>
RangeIndex: 1949 entries, 0 to 1948
Data columns (total 11 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Country name                      1949 non-null   str    
 1   year                              1949 non-null   int64  
 2   Life Ladder                       1949 non-null   float64
 3   Log GDP per capita                1913 non-null   float64
 4   Social support                    1936 non-null   float64
 5   Healthy life expectancy at birth  1894 non-null   float64
 6   Freedom to make life choices      1917 non-null   float64
 7   Generosity                        1860 non-null   float64
 8   Perceptions of corruption         1839 non-null   float64
 9   Positive affect                   1927 non-null   float64
 10  Negative affect                   1933 non-null   float64
dtypes: float64(9), int64(1), str(1)
m

### Passo 2: Diagnóstico de Qualidade dos Dados (Camada Raw)
Mapeamento da saúde da base: verificação de duplicatas e porcentagem de valores nulos por coluna.

In [51]:
# 1. Checando linhas totalmente duplicadas
total_duplicatas = df_raw.duplicated().sum()
print(f"Total de linhas duplicadas: {total_duplicatas}")
print("-" * 40)

# 2. Calculando a quantidade e a porcentagem de valores nulos por coluna
print("\n--- Diagnóstico de Valores Nulos ---")
nulos_qtd = df_raw.isnull().sum()
nulos_pct = (nulos_qtd / len(df_raw)) * 100

df_nulos = pd.DataFrame({
    'Quantidade de Nulos': nulos_qtd,
    'Porcentagem (%)': nulos_pct.round(2)
})

df_nulos = df_nulos[df_nulos['Quantidade de Nulos'] > 0].sort_values(by='Porcentagem (%)', ascending=False)
display(df_nulos)

Total de linhas duplicadas: 0
----------------------------------------

--- Diagnóstico de Valores Nulos ---


,Quantidade de Nulos,Porcentagem (%)
Perceptions of corruption,110,5.64
Generosity,89,4.57
Healthy life expectancy at birth,55,2.82
Log GDP per capita,36,1.85
Freedom to make life choices,32,1.64
Positive affect,22,1.13
Negative affect,16,0.82
Social support,13,0.67


### Passo 2.5: Mapeamento de Nulos por País e Análise Profunda
Identificando os maiores ofensores de dados faltantes e analisando se as falhas são pontuais (um ano isolado) ou estruturais (vários anos seguidos em branco).

In [50]:
# 1. Panorama geral de países afetados
linhas_com_nulos = df_raw[df_raw.isnull().any(axis=1)]
contagem_nulos_por_pais = linhas_com_nulos['Country name'].value_counts()

total_paises = df_raw['Country name'].nunique()
paises_afetados = len(contagem_nulos_por_pais)

print(f"Panorama: Dos {total_paises} países presentes na base, {paises_afetados} possuem algum dado faltante.")
print("\n--- Top 15 Países com mais registros incompletos ---")

df_top_incompletos = contagem_nulos_por_pais.reset_index()
df_top_incompletos.columns = ['País', 'Qtd de Anos com Dados Faltantes']
display(df_top_incompletos.head(15))

# 2. Matriz de Nulos por Coluna para os Top 5 países com mais falhas
# Pegamos o nome dos 5 países do topo da lista
top_5_paises = contagem_nulos_por_pais.head(5).index

# Filtramos a base original apenas para esses 5 países
df_top5_incompletos = df_raw[df_raw['Country name'].isin(top_5_paises)]

# Criamos a matriz: definimos o país como índice, verificamos onde é nulo e somamos por país
df_matriz_nulos = df_top5_incompletos.set_index('Country name').isnull().groupby('Country name').sum().astype(int)

# Opcional: Remover a coluna 'year' da visualização caso ela não tenha nenhum nulo (para deixar a tabela mais limpa)
if 'year' in df_matriz_nulos.columns and df_matriz_nulos['year'].sum() == 0:
    df_matriz_nulos = df_matriz_nulos.drop(columns=['year'])

print(f"\n--- Distribuição de dados faltantes (nulos) por métrica nos Top 5 países ---")
display(df_matriz_nulos)

Panorama: Dos 166 países presentes na base, 74 possuem algum dado faltante.

--- Top 15 Países com mais registros incompletos ---


,País,Qtd de Anos com Dados Faltantes
0,China,15
1,Kosovo,14
2,Jordan,13
3,Saudi Arabia,13
4,Hong Kong S.A.R. of China,11
5,Palestinian Territories,10
6,Taiwan Province of China,10
7,Turkmenistan,10
8,United Arab Emirates,10
9,Bahrain,8



--- Distribuição de dados faltantes (nulos) por métrica nos Top 5 países ---


,Life Ladder,Log GDP per capita,Social support,Healthy life expectancy at birth,Freedom to make life choices,Generosity,Perceptions of corruption,Positive affect,Negative affect
Country name,,,,,,,,,
China,0,0,0,0,5,1,15,0,0
Hong Kong S.A.R. of China,0,1,0,11,0,1,0,0,0
Jordan,0,0,0,0,2,1,11,3,3
Kosovo,0,1,0,14,1,1,0,1,0
Saudi Arabia,0,0,0,0,1,1,12,0,0


### Passo 3: Limpeza e Tratamento Condicional (Camada Silver)
Aplicação da interpolação linear condicional. Preenchemos as lacunas matemáticas apenas se o país tiver menos de 20% de dados faltantes na métrica específica ao longo dos anos.

In [53]:
# 1. Criando a base Silver a partir da Raw
df_silver = df_raw.copy()

# 2. Ordenando cronologicamente por país
df_silver = df_silver.sort_values(by=['Country name', 'year']).reset_index(drop=True)

# 3. Separando as colunas de métricas
colunas_metricas = df_silver.select_dtypes(include=['float64', 'int64']).columns.drop('year', errors='ignore')

# 4. Função inteligente de interpolação condicional
def interpolacao_inteligente(serie_temporal):
    total_registros = len(serie_temporal)
    total_nulos = serie_temporal.isnull().sum()
    
    # Se a quantidade de nulos for maior que a metade dos registros, retorna sem interpolar
    if total_nulos > (total_registros / 5):
        return serie_temporal
    # Caso contrário, aplica a interpolação linear
    else:
        return serie_temporal.interpolate(method='linear', limit_direction='both')

# 5. Aplicando a função agrupada por país e por coluna
for coluna in colunas_metricas:
    df_silver[coluna] = df_silver.groupby('Country name')[coluna].transform(interpolacao_inteligente)

print("Tratamento da camada Silver concluído com sucesso!")

Tratamento da camada Silver concluído com sucesso!


### Passo 3.5: Validação da Regra de Confiança
Conferindo quantos nulos foram preenchidos e isolando o histórico dos países que tiveram seus dados nulos preservados de propósito para não enviesar a análise visual.

In [54]:
print("--- Nulos Preservados após Regra de Limite de Confiança ---")
nulos_restantes = df_silver[colunas_metricas].isnull().sum()
nulos_final = nulos_restantes[nulos_restantes > 0]

if nulos_final.empty:
    print("Todos os nulos foram preenchidos (Nenhum país estourou o limite de 20%).")
else:
    print("Estes são os nulos que sobraram DE PROPÓSITO para não enviesar a visualização:")
    df_nulos_preservados = pd.DataFrame({
        'Quantidade de Nulos Retidos': nulos_final,
        'Porcentagem do Total da Base (%)': ((nulos_final / len(df_silver)) * 100).round(2)
    }).sort_values(by='Quantidade de Nulos Retidos', ascending=False)
    display(df_nulos_preservados)

# Histórico Completo dos Países com Nulos Preservados
linhas_com_nulos_silver = df_silver[df_silver.isnull().any(axis=1)]
paises_com_nulos_restantes = linhas_com_nulos_silver['Country name'].unique()

print(f"\nTotal de países que mantiveram algum nulo: {len(paises_com_nulos_restantes)}")
print(f"Países: {list(paises_com_nulos_restantes)}")
print("-" * 50)

df_historico_restantes = df_silver[df_silver['Country name'].isin(paises_com_nulos_restantes)]

pd.set_option('display.max_rows', None)
display(df_historico_restantes)
pd.reset_option('display.max_rows')

--- Nulos Preservados após Regra de Limite de Confiança ---
Estes são os nulos que sobraram DE PROPÓSITO para não enviesar a visualização:


,Quantidade de Nulos Retidos,Porcentagem do Total da Base (%)
Perceptions of corruption,100,5.13
Healthy life expectancy at birth,55,2.82
Generosity,46,2.36
Log GDP per capita,25,1.28
Freedom to make life choices,12,0.62
Positive affect,5,0.26
Social support,4,0.21
Negative affect,4,0.21



Total de países que mantiveram algum nulo: 28
Países: ['Algeria', 'Bahrain', 'China', 'Cuba', 'Djibouti', 'Egypt', 'Hong Kong S.A.R. of China', 'Iran', 'Jordan', 'Kosovo', 'Kuwait', 'Libya', 'Maldives', 'Malta', 'North Cyprus', 'Oman', 'Palestinian Territories', 'Qatar', 'Saudi Arabia', 'Somalia', 'Somaliland region', 'South Sudan', 'Taiwan Province of China', 'Turkmenistan', 'United Arab Emirates', 'Venezuela', 'Vietnam', 'Yemen']
--------------------------------------------------


,Country name,year,Life Ladder,Log GDP per capita,Social support,Healthy life expectancy at birth,Freedom to make life choices,Generosity,Perceptions of corruption,Positive affect,Negative affect
25,Algeria,2010,5.464,9.287,0.8100,64.500,0.5930,-0.2050,0.618,0.550000,0.2550
26,Algeria,2011,5.317,9.297,0.8100,64.660,0.5300,-0.1810,0.638,0.550000,0.2550
27,Algeria,2012,5.605,9.311,0.8390,64.820,0.5870,-0.1720,0.690,0.604000,0.2300
28,Algeria,2014,6.355,9.335,0.8180,65.140,NaN,NaN,NaN,0.626000,0.1770
29,Algeria,2016,5.341,9.362,0.7490,65.500,NaN,NaN,NaN,0.661000,0.3770
30,Algeria,2017,5.249,9.354,0.8070,65.700,0.4370,-0.1670,0.700,0.642000,0.2890
31,Algeria,2018,5.043,9.348,0.7990,65.900,0.5830,-0.1460,0.759,0.591000,0.2930
32,Algeria,2019,4.745,9.337,0.8030,66.100,0.3850,0.0050,0.741,0.585000,0.2150
107,Bahrain,2009,5.701,10.709,0.9040,65.940,0.8960,0.0370,0.506,0.764000,0.4220
108,Bahrain,2010,5.937,10.706,0.8770,66.300,0.8620,-0.0010,0.715,0.685000,0.4230


### Passo 4: Tradução, Integração com IDH e Exportação (Camada Gold)
Padronização das colunas para Português, ingestão do JSON da ONU contendo o IDH e merge (cruzamento) dos dados (Left Join) para enriquecimento da base final.

In [55]:
# 1. Tradução das Colunas para Português
dicionario_colunas_pt = {
    'Country name': 'Pais',
    'year': 'Ano',
    'Life Ladder': 'Escala_Vida',
    'Log GDP per capita': 'Log_PIB_Per_Capita',
    'Social support': 'Suporte_Social',
    'Healthy life expectancy at birth': 'Expectativa_Vida_Saudavel',
    'Freedom to make life choices': 'Liberdade_Escolhas',
    'Generosity': 'Generosidade',
    'Perceptions of corruption': 'Percepcao_Corrupcao',
    'Positive affect': 'Emocoes_Positivas',
    'Negative affect': 'Emocoes_Negativas'
}

df_gold = df_silver.copy().rename(columns=dicionario_colunas_pt)
colunas_float = df_gold.select_dtypes(include=['float64']).columns
df_gold[colunas_float] = df_gold[colunas_float].round(4)

# 2. Ingestão e tratamento da base de IDH
print("Carregando e processando o JSON do IDH...")
df_hdr = pd.read_json('Data/hdr-data.json')
df_idh = df_hdr[df_hdr['indicatorCode'] == 'hdi'].copy()

df_idh = df_idh[['country', 'year', 'value']].rename(columns={
    'country': 'Pais',
    'year': 'Ano',
    'value': 'IDH'
})

df_idh['Ano'] = df_idh['Ano'].astype(int)
df_idh['IDH'] = pd.to_numeric(df_idh['IDH'], errors='coerce')

# Tradutor de países ONU -> Relatório de Felicidade
substituicoes_paises = {
    'Russian Federation': 'Russia', 'Korea (Republic of)': 'South Korea', 'Türkiye': 'Turkey',
    'Bolivia (Plurinational State of)': 'Bolivia', 'Venezuela (Bolivarian Republic of)': 'Venezuela',
    'Iran (Islamic Republic of)': 'Iran', 'Viet Nam': 'Vietnam', 'Moldova (Republic of)': 'Moldova',
    'Tanzania (United Republic of)': 'Tanzania', 'Syrian Arab Republic': 'Syria',
    'Congo (Democratic Republic of the)': 'Congo (Kinshasa)', 'Congo': 'Congo (Brazzaville)'
}
df_idh['Pais'] = df_idh['Pais'].replace(substituicoes_paises)

# 3. Cruzamento e Exportação
print("Cruzando dados de Felicidade com IDH...")
df_gold = pd.merge(df_gold, df_idh, on=['Pais', 'Ano'], how='left')
df_gold['IDH'] = df_gold['IDH'].round(3)

print("--- Amostra da Base Gold Final com IDH ---")
display(df_gold.head(3))

nome_arquivo_final = 'world_happiness_gold_pt.csv'
df_gold.to_csv(nome_arquivo_final, index=False, encoding='utf-8')
print(f"\n✅ Sucesso absoluto! O arquivo '{nome_arquivo_final}' foi exportado.")

Carregando e processando o JSON do IDH...
Cruzando dados de Felicidade com IDH...
--- Amostra da Base Gold Final com IDH ---


,Pais,Ano,Escala_Vida,Log_PIB_Per_Capita,Suporte_Social,Expectativa_Vida_Saudavel,Liberdade_Escolhas,Generosidade,Percepcao_Corrupcao,Emocoes_Positivas,Emocoes_Negativas,IDH
0,Afghanistan,2008,3.724,7.370,0.451,50.8,0.718,0.168,0.882,0.518,0.258,0.446
1,Afghanistan,2009,4.402,7.540,0.552,51.2,0.679,0.190,0.850,0.584,0.237,0.458
2,Afghanistan,2010,4.758,7.647,0.539,51.6,0.600,0.121,0.707,0.618,0.275,0.465



✅ Sucesso absoluto! O arquivo 'world_happiness_gold_pt.csv' foi exportado.
